In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery


### 1. Weekly Cohort Retention

In [3]:
orders["order_week"] = orders["order_ts"].dt.to_period("W").dt.start_time ## Date Trunc equivalent
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_week
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy,2024-12-30
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy,2025-01-06
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy,2025-01-06
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy,2025-01-06
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery,2025-01-06


In [4]:
orders.dtypes

order_id                 int64
user_id                  int64
store_id                   str
city                       str
order_ts        datetime64[us]
items                    int64
gmv                    float64
delivery_min           float64
category                   str
order_week      datetime64[us]
dtype: object

In [5]:
user_order_weeks = orders[['user_id', 'order_week']].drop_duplicates()
user_order_weeks["user_first_week"] = (
    user_order_weeks
    .groupby(["user_id"])["order_week"]
    .transform("min")
)

user_order_weeks["week_diff"] = (
    user_order_weeks["order_week"] - user_order_weeks["user_first_week"]
)/pd.Timedelta(weeks=1)

user_order_weeks.head()

,user_id,order_week,user_first_week,week_diff
0,10700,2024-12-30,2024-12-30,0.0
1,8110,2025-01-06,2025-01-06,0.0
2,15650,2025-01-06,2025-01-06,0.0
3,1908,2025-01-06,2025-01-06,0.0
4,2751,2025-01-06,2025-01-06,0.0


In [6]:
pivot_df = pd.pivot_table(
    user_order_weeks,
    index = "user_first_week",
    columns = "week_diff",
    values = "user_id",
    aggfunc = "count"
)

pivot_df = pivot_df.div(pivot_df[0], axis=0) * 100

pivot_df

week_diff,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,...,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0
user_first_week,,,,,,,,,,,,,,,,,,,,,
2024-12-30,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,100.000000,NaN,100.000000,NaN,NaN,NaN,NaN
2025-01-06,100.0,NaN,6.250000,NaN,NaN,12.500000,18.750000,NaN,6.250000,6.250000,...,NaN,NaN,NaN,NaN,6.250000,NaN,NaN,6.250000,12.500000,12.500000
2025-01-13,100.0,7.407407,14.814815,14.814815,7.407407,11.111111,NaN,18.518519,3.703704,3.703704,...,14.814815,11.111111,NaN,3.703704,7.407407,NaN,3.703704,3.703704,7.407407,3.703704
2025-01-20,100.0,10.810811,8.108108,13.513514,10.810811,10.810811,2.702703,5.405405,8.108108,16.216216,...,10.810811,2.702703,13.513514,5.405405,18.918919,5.405405,10.810811,10.810811,5.405405,NaN
2025-01-27,100.0,2.083333,4.166667,6.250000,8.333333,2.083333,4.166667,8.333333,8.333333,4.166667,...,10.416667,8.333333,6.250000,14.583333,6.250000,4.166667,8.333333,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-31,100.0,8.988764,6.741573,11.235955,1.123596,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-07,100.0,4.705882,4.705882,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-14,100.0,2.666667,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. 7-day rolling GMV per city

In [7]:
orders["order_date"] = orders["order_ts"].dt.normalize()

city_summary = (
    orders
    .groupby(["city"], as_index = False)
    .agg(
        first_order_date = ("order_date", "min"),
        last_order_date = ("order_date", "max")
    )
)

city_order_dates = pd.concat(
    city_summary.apply(
        lambda row: pd.DataFrame({
            "city": row["city"],
            "order_date": pd.date_range(
                row["first_order_date"],
                row["last_order_date"],
                freq="D"
            )
        }),
        axis=1
    ).tolist(),
    ignore_index=True
)

daily_city_summary = (
    orders
    .groupby(["order_date", "city"], as_index = False)
    .agg(
        gmv = ("gmv", "sum")
    )
)

daily_city_summary = city_order_dates.merge(
    daily_city_summary,
    on = ["city", "order_date"],
    how = "left"
)

daily_city_summary["gmv"] = daily_city_summary["gmv"].fillna(0)

daily_city_summary["rolling_7d_gmv"] = (
    daily_city_summary
    .groupby("city")["gmv"]
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

daily_city_summary

,city,order_date,gmv,rolling_7d_gmv
0,Bengaluru,2025-01-08,199.17,NaN
1,Bengaluru,2025-01-09,0.00,NaN
2,Bengaluru,2025-01-10,0.00,NaN
3,Bengaluru,2025-01-11,164.65,NaN
4,Bengaluru,2025-01-12,573.28,NaN
...,...,...,...,...
3783,Pune,2026-09-25,14823.65,109576.38
3784,Pune,2026-09-26,12700.97,111703.24
3785,Pune,2026-09-27,9970.62,106610.41
3786,Pune,2026-09-28,6288.67,95112.12


Here we have created a base of order dates and cities. If we hadn't created that, we would need to use rolling("7D"). Final outcome would only change where there is no order date present in a city. Since, order date is missing, final output will miss that date. But where dates are present, rolling("7D") would have worked same for SUM

### 3. Order Sequence per User

In [8]:
orders["order_number"] = (
    orders
    .groupby(["user_id"])["order_ts"]
    .rank(
        method = "first",
        ascending = True
    )
)

orders["prev_order_ts"] = (
    orders
    .sort_values(["user_id", "order_ts"])
    .groupby("user_id")["order_ts"]
    .shift(1)
)

orders["days_since_prev_order"] = (
    orders["order_ts"] - orders["prev_order_ts"]
).dt.days

orders["prev_order_gmv"] = (
    orders
    .sort_values(["user_id", "order_ts"])
    .groupby("user_id")["gmv"]
    .shift(1)
)

orders["is_higher_gmv_than_prev_order"] = (
    orders["gmv"] > orders["prev_order_gmv"]
)

orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_week,order_date,order_number,prev_order_ts,days_since_prev_order,prev_order_gmv,is_higher_gmv_than_prev_order
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy,2024-12-30,2025-01-02,1.0,NaT,NaN,NaN,False
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy,2025-01-06,2025-01-06,1.0,NaT,NaN,NaN,False
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy,2025-01-06,2025-01-06,1.0,NaT,NaN,NaN,False
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy,2025-01-06,2025-01-08,1.0,NaT,NaN,NaN,False
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery,2025-01-06,2025-01-08,1.0,NaT,NaN,NaN,False


In [9]:
second_orders = orders[orders["order_number"] == 2]

second_orders["days_since_prev_order"].median()

np.float64(38.0)